# UrbanPulse Nepal — Temporal Pattern Analysis

This notebook describes when incidents in the retrieved BIPAD snapshot were recorded. It examines year-by-year and month-by-month counts, season groupings, and hazard-ID distributions. Calendar features are for historical description only. Counts can reflect reporting coverage and administrative practices as well as real incidents; they are not population-adjusted rates or forecasts.

Run notebooks 01 and 02 first. This notebook reads the processed snapshot, highlights partial years and months with no records, and saves temporal summary tables for later reporting and dashboard work.

## 1. Load incident dates and preserve the coverage caveat

Dates are interpreted in Nepal time. The first and last observed calendar years are partial by construction; 2026 coverage in this snapshot ends on October 13. A month with no records is reported as an empty observed month, not as proof that no disaster occurred.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'data').is_dir(): PROJECT_ROOT=PROJECT_ROOT.parent
if not (PROJECT_ROOT/'data').is_dir(): raise FileNotFoundError('Run from repository or notebooks/ folder.')
DATA_ROOT=PROJECT_ROOT/'data'/'processed'/'bipad_incidents'
data_dirs=sorted((p for p in DATA_ROOT.iterdir() if (p/'incidents_processed.csv').is_file() and (p/'data_quality_report.json').is_file()),key=lambda p:p.name)
if not data_dirs: raise FileNotFoundError('Run notebooks 01 and 02 first.')
DATA_DIR=data_dirs[-1]
df=pd.read_csv(DATA_DIR/'incidents_processed.csv',low_memory=False)
quality=json.loads((DATA_DIR/'data_quality_report.json').read_text(encoding='utf-8'))
df['incident_datetime']=pd.to_datetime(df['incidentOn_nepal'],errors='coerce',utc=True).dt.tz_convert('Asia/Kathmandu')
df=df.dropna(subset=['incident_datetime']).copy()
df['incident_month_period']=df['incident_datetime'].dt.to_period('M')
df['incident_year']=df['incident_datetime'].dt.year.astype(int)
df['month_number']=df['incident_datetime'].dt.month.astype(int)
df['hazard_id']=df['hazard_id'].fillna('unknown').astype('string')
if 'incident_season' not in df.columns: raise ValueError('Run notebook 02 with seasonal feature creation first.')
print('Snapshot:',DATA_DIR.name)
print('Valid incident dates:',f'{len(df):,}')
print('Observed period:',df['incident_datetime'].min().isoformat(),'to',df['incident_datetime'].max().isoformat())
print('Source pagination complete:',quality.get('quality_audit',{}).get('pagination_complete'))

## 2. Summarize annual and monthly record counts

Year totals are useful for describing the snapshot, but must be read with the partial-year flags and the unverified historical coverage. The 12-month rolling mean below smooths the reported monthly counts; it is not a statistical trend estimate and does not adjust for exposure or data-collection changes.

In [ ]:
yearly=df.groupby('incident_year').size().rename('incident_records').reset_index()
yearly['coverage_note']='Full calendar year in date range'
yearly.loc[yearly['incident_year']==df['incident_year'].min(),'coverage_note']='First observed year may be partial'
yearly.loc[yearly['incident_year']==df['incident_year'].max(),'coverage_note']='Last observed year is partial'
monthly=df.groupby('incident_month_period').size().rename('incident_records').to_frame()
full_months=pd.period_range(monthly.index.min(),monthly.index.max(),freq='M')
monthly=monthly.reindex(full_months,fill_value=0).rename_axis('month').reset_index()
monthly['month_start']=monthly['month'].dt.to_timestamp()
monthly['rolling_12_month_average']=monthly['incident_records'].rolling(12,min_periods=12).mean()
monthly['no_records_in_snapshot']=(monthly['incident_records']==0)
missing_months=monthly.loc[monthly['no_records_in_snapshot'],'month'].astype(str).tolist()
print('Annual records:'); display(yearly)
print('Number of zero-record months within observed date span:',len(missing_months))
if missing_months: print('Zero-record month examples:',missing_months[:20])
fig,axes=plt.subplots(2,1,figsize=(12,8),sharex=False)
axes[0].bar(yearly['incident_year'],yearly['incident_records'],color='#18794e')
axes[0].set(title='Incident records by incident year',xlabel='Year',ylabel='Records')
axes[1].plot(monthly['month_start'],monthly['incident_records'],linewidth=0.8,label='Monthly records',color='#2f855a')
axes[1].plot(monthly['month_start'],monthly['rolling_12_month_average'],linewidth=1.8,label='12-month rolling average',color='#253c2f')
axes[1].set(title='Monthly incident records in retrieved snapshot',xlabel='Month',ylabel='Records'); axes[1].legend()
for ax in axes: ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

## 3. Compare seasonal and within-year patterns

The notebook uses the season labels derived in notebook 02: Winter (December–February), Pre-monsoon (March–May), Monsoon (June–September), and Post-monsoon (October–November). A simple month-of-year count sums across all observed years and can over-represent years with better coverage or more recorded incidents.

In [ ]:
season_order=['Winter','Pre-monsoon','Monsoon','Post-monsoon']
seasonal=df.groupby('incident_season',dropna=False).size().reindex(season_order,fill_value=0).rename('incident_records').reset_index()
seasonal['share_of_records']=seasonal['incident_records']/len(df)
month_of_year=df.groupby('month_number').size().reindex(range(1,13),fill_value=0).rename('incident_records').reset_index()
month_of_year['month_name']=pd.to_datetime(month_of_year['month_number'],format='%m').dt.strftime('%b')
display(seasonal); display(month_of_year)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(seasonal['incident_season'],seasonal['incident_records'],color='#20835d'); axes[0].tick_params(axis='x',rotation=25)
axes[0].set(title='Records by derived season',ylabel='Records')
axes[1].bar(month_of_year['month_name'],month_of_year['incident_records'],color='#68a987')
axes[1].set(title='Records by calendar month (all observed years)',xlabel='Month',ylabel='Records')
for ax in axes: ax.grid(axis='y',alpha=0.2)
plt.tight_layout(); plt.show()

## 4. Inspect temporal patterns by hazard ID

Hazard values remain BIPAD IDs because readable labels have not been joined in the saved incident data. The charts report counts for the most frequent IDs in this snapshot; they should not be interpreted as comparable hazard rates without a verified complete source and appropriate exposure denominators.

In [ ]:
top_hazards=df['hazard_id'].value_counts().head(8).index.tolist()
hazard_year=pd.crosstab(df['incident_year'],df['hazard_id'])
hazard_year=hazard_year.reindex(columns=top_hazards,fill_value=0)
display(hazard_year.tail(10))
hazard_season=pd.crosstab(df['hazard_id'],df['incident_season'],normalize='index').reindex(columns=season_order,fill_value=0)
hazard_season['total_records']=df['hazard_id'].value_counts().reindex(hazard_season.index).values
display(hazard_season.sort_values('total_records',ascending=False).head(12))
ax=hazard_year.plot(kind='line',marker='.',figsize=(12,5))
ax.set(title='Annual records for the eight most frequent hazard IDs',xlabel='Incident year',ylabel='Records'); ax.grid(alpha=0.2); ax.legend(title='Hazard ID',ncol=2)
plt.tight_layout(); plt.show()

## 5. Save temporal summaries and coverage diagnostics

The output tables preserve counts and calendar grouping for later visualization. The coverage report distinguishes observed zero-record months from months outside the first and last incident dates. Because archive completeness is unverified, a zero-record month cannot be confidently interpreted as no disasters having occurred.

In [ ]:
OUTPUT_DIR=PROJECT_ROOT/'data'/'processed'/'temporal'/DATA_DIR.name
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
yearly_path=OUTPUT_DIR/'annual_incident_counts.csv'
monthly_path=OUTPUT_DIR/'monthly_incident_counts.csv'
seasonal_path=OUTPUT_DIR/'seasonal_summary.csv'
hazard_year_path=OUTPUT_DIR/'annual_hazard_id_counts.csv'
yearly.to_csv(yearly_path,index=False,encoding='utf-8-sig')
monthly.assign(month=monthly['month'].astype(str)).to_csv(monthly_path,index=False,encoding='utf-8-sig')
seasonal.to_csv(seasonal_path,index=False,encoding='utf-8-sig')
hazard_year.to_csv(hazard_year_path,encoding='utf-8-sig')
temporal_report={'snapshot_id':DATA_DIR.name,'incident_records_with_valid_dates':len(df),'date_min_nepal':df['incident_datetime'].min().isoformat(),'date_max_nepal':df['incident_datetime'].max().isoformat(),'first_observed_year':int(df['incident_year'].min()),'last_observed_year':int(df['incident_year'].max()),'last_year_partial':True,'zero_record_months_inside_observed_date_span':missing_months,'source_pagination_complete':quality.get('quality_audit',{}).get('pagination_complete'),'coverage_limitation':quality.get('coverage_limitation'),'interpretation':'Counts describe this retrieved snapshot, not complete disaster incidence or future risk.'}
report_path=OUTPUT_DIR/'temporal_coverage_report.json'
report_path.write_text(json.dumps(temporal_report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved annual counts:',yearly_path)
print('Saved monthly counts:',monthly_path)
print('Saved seasonal summary:',seasonal_path)
print('Saved hazard-year counts:',hazard_year_path)
print('Saved coverage report:',report_path)

## Findings from the executed temporal analysis

After running this notebook, add the measured annual, monthly, seasonal, hazard-ID, and zero-record-month findings here. State which years are partial, describe the incomplete-coverage caveat, and avoid interpreting record counts as changing disaster risk.

This final findings section is reserved for results from the saved output of this notebook.